# U-Net for Pancreas Segmentation in Abdominal CT Scans
- 2D U-Net for segmenting the pancreas from contrast-enhanced 3D abdominal CT scans (TCIA Pancreas-CT Dataset).

- **Hounsfield Unit (HU) Windowing:** Clips CT intensity values to `[-125, 225]` for pancreas soft-tissue contrast.
- **Dice + BCE Loss:** Handles high class imbalance (pancreas represents $<1.5\%$ of voxels).
- **Mixed Precision (AMP):** Faster training with lower GPU memory footprint.
- **Google Drive Checkpointing:** Saves `best_model{time}.pth` directly to Google Drive.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Install Dependencies & Set Working Directory
!pip install -q monai nibabel SimpleITK tqdm matplotlib

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

In [ ]:
# 3. Change directory to project root
%cd /content/drive/MyDrive/Pancreas-Seg-MIP
!ls -la

In [ ]:
# 4. Launch Training Loop
!python train.py \
  --data_dir ./data \
  --output_dir ./checkpoints \
  --save_drive_path "/content/drive/MyDrive/Pancreas_Checkpoints" \
  --epochs 25 \
  --batch_size 8 \
  --lr 1e-4

In [ ]:
# 5. Visualize Ground Truth vs Prediction on a Sample CT Slice
import torch
from model import build_model
from dataset import Pancreas2DDataset
from utils import plot_prediction_overlay
import glob

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model('2d').to(device)

# Load best checkpoint
checkpoint = torch.load('/content/drive/MyDrive/Pancreas_Checkpoints/best_model.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Load sample batch
val_images = sorted(glob.glob('./data/images/*.nii*'))
val_masks = sorted(glob.glob('./data/labels/*.nii*'))

if val_images:
    dataset = Pancreas2DDataset(val_images[:2], val_masks[:2])
    img, mask = dataset[0]
    img_tensor = img.unsqueeze(0).to(device)
    
    with torch.no_grad():
        pred = torch.sigmoid(model(img_tensor))
    
    plot_prediction_overlay(
        ct_slice=img.squeeze(),
        ground_truth=mask.squeeze(),
        prediction=pred.squeeze(),
        title=f"Val Dice Score: {checkpoint['val_dice']:.4f}"
    )